# Урок 14 · Задания: нейросеть на MNIST

**Что делаем сегодня:** обучаем сеть распознавать рукописные цифры, добиваемся точности выше 0,95, находим её ошибки и разбираемся, почему сеть иногда «уверенно врёт».

> 🧭 **Как работать с ноутбуком:** запускай ячейки по порядку кнопкой ▶ (или `Shift+Enter`). Под каждым блоком кода написано, **что должно получиться**. Не спеши — читай пояснения.

---


## 🔧 Шаг 0. Подключаем инструменты

Сначала импортируем библиотеки. `tensorflow` — для нейросети, `numpy` — для чисел, `matplotlib` — чтобы рисовать картинки.

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense

print("Всё готово. Версия TensorFlow:", tf.__version__)

## 📦 Шаг 1. Загружаем данные MNIST

MNIST — это 70 000 картинок рукописных цифр (0–9), размером 28×28 пикселей.
60 000 для обучения (train) и 10 000 для проверки (test).

**Важная строка — деление на 255.** Каждый пиксель хранит яркость от 0 (чёрный) до 255 (белый).
Делим на 255, чтобы все числа стали от 0 до 1 — так сеть учится быстрее и стабильнее. Это называется **нормализация**.

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Нормализация: значения пикселей 0..255  ->  0..1
X_train = X_train / 255.0
X_test  = X_test  / 255.0

print("Картинок для обучения:", X_train.shape[0])
print("Картинок для проверки:", X_test.shape[0])
print("Размер одной картинки:", X_train.shape[1], "x", X_train.shape[2])

### 👀 Посмотрим на данные глазами

Прежде чем обучать — всегда полезно взглянуть, с чем работаем. Покажем первые 8 картинок и их правильные ответы.

In [ ]:
plt.figure(figsize=(12, 3))
for i in range(8):
    plt.subplot(1, 8, i + 1)
    plt.imshow(X_train[i], cmap="gray")
    plt.title(f"Это {y_train[i]}")
    plt.axis("off")
plt.show()

**🔍 Как сеть «видит» эту картинку?**

Не как рисунок, а как **таблицу чисел** 28×28. Давай убедимся — распечатаем одну цифру числами (округлённо, чтобы влезло).

In [ ]:
# Берём первую картинку и печатаем её как таблицу чисел
img = X_train[0]
for row in img:
    line = "".join(f"{int(v*9)}" if v > 0.1 else "." for v in row)
    print(line)

print("\nПравильный ответ:", y_train[0])
print("Для сети это просто 28*28 =", 28*28, "чисел, а не 'изображение'.")

---
# 🟢 Базовый уровень

**Задание:** обучи сеть и добейся точности выше 0,95 на тесте. Затем найди 3 картинки, на которых сеть ошиблась, и реши — трудные они или ошибки странные.

## Шаг 2. Собираем нейросеть

Наша сеть из трёх слоёв:
- `Flatten` — разворачивает картинку 28×28 в один ряд из 784 чисел (сеть не умеет смотреть на квадрат);
- `Dense(128, relu)` — 128 нейронов, которые ищут узоры;
- `Dense(10, softmax)` — 10 нейронов, по одному на каждую цифру. `softmax` превращает ответы в вероятности.

In [ ]:
model = Sequential([
    Flatten(input_shape=(28, 28)),      # картинка -> ряд из 784 чисел
    Dense(128, activation="relu"),      # скрытый слой
    Dense(10,  activation="softmax"),   # 10 цифр -> вероятности
])

# Настройки обучения (пока просто копируем — это шаблон)
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

model.summary()   # покажет структуру сети

## Шаг 3. Обучаем

`fit` запускает обучение. `epochs=5` — сеть пройдёт все картинки 5 раз, каждый раз чуть подкручивая веса (это backpropagation).

> ⏳ Займёт примерно полминуты. Смотри, как **accuracy** растёт с каждой эпохой.

In [ ]:
model.fit(X_train, y_train, epochs=5, verbose=1)

## Шаг 4. Проверяем точность на тесте

Это честная проверка — на картинках, которых сеть **не видела** при обучении.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Точность на тесте: {test_acc:.4f}")

if test_acc > 0.95:
    print("✅ Задание выполнено: точность выше 0,95!")
else:
    print("Попробуй увеличить epochs — точность подрастёт.")

## Шаг 5. 🔎 Находим ошибки сети

Сеть ошибается редко (~2-3 картинки из 100), но такие есть. Найдём их: сравним ответ сети с правильным.

In [ ]:
# probs — вероятности по всем 10 цифрам для каждой картинки теста
probs = model.predict(X_test, verbose=0)
preds = probs.argmax(axis=1)          # цифра с самой большой вероятностью = ответ сети

# Индексы картинок, где сеть ошиблась
wrong = np.where(preds != y_test)[0]
print(f"Сеть ошиблась на {len(wrong)} картинках из {len(y_test)}")

### Показываем 3 ошибки

Под каждой картинкой: что сказала сеть, **насколько она была уверена**, и какой правильный ответ.

In [ ]:
plt.figure(figsize=(11, 4))
for i, idx in enumerate(wrong[:3]):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_test[idx], cmap="gray")
    confidence = probs[idx].max() * 100
    plt.title(f"Сеть: {preds[idx]}  ({confidence:.0f}%)\nПравда: {y_test[idx]}",
              fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

### 🤔 Твой вывод

Посмотри на свои 3 картинки и ответь **словами** (запиши прямо в ячейку ниже):

- Эти цифры реально трудные (криво написаны, сам бы засомневался)?
- Или ошибки странные — цифра читается легко, а сеть промахнулась?

Оба ответа правильные — задание про твоё рассуждение.

In [ ]:
# 📝 Напиши свой вывод здесь (можно прямо в кавычках):
moy_vyvod = """
...
"""
print(moy_vyvod)

### ⭐ Бонус: самые «самоуверенные» ошибки

Самое интересное — когда сеть ошиблась, но была уверена на 90%+. Покажем именно такие. Это подводит к главному вопросу урока про доверие ИИ.

In [ ]:
conf_wrong = probs[wrong].max(axis=1)             # уверенность на каждой ошибке
worst = wrong[conf_wrong.argsort()[::-1][:3]]     # 3 самые уверенные ошибки

plt.figure(figsize=(11, 4))
for i, idx in enumerate(worst):
    plt.subplot(1, 3, i + 1)
    plt.imshow(X_test[idx], cmap="gray")
    confidence = probs[idx].max() * 100
    plt.title(f"Сеть уверена: {preds[idx]} ({confidence:.0f}%)\nА правда: {y_test[idx]}",
              fontsize=12)
    plt.axis("off")
plt.tight_layout()
plt.show()

print("Сеть БЫЛА УВЕРЕНА и всё равно ошиблась. Запомни это чувство — вернёмся к нему.")

---
# ⭐ Со звёздочкой: эксперименты с архитектурой

**Задание:** добавь второй скрытый слой или увеличь число эпох. Как меняется точность? Найди момент, когда точность на train растёт, а на test — нет. Это **переобучение** из урока 7, теперь на нейросети.

## Шаг 6. Новая сеть — два скрытых слоя, 15 эпох

Добавили `Dense(64)` и учим дольше. `validation_data` позволяет видеть точность на train **и** на test после каждой эпохи — именно это нам нужно, чтобы поймать переобучение.

In [ ]:
model2 = Sequential([
    Flatten(input_shape=(28, 28)),
    Dense(128, activation="relu"),
    Dense(64,  activation="relu"),      # <- добавленный второй скрытый слой
    Dense(10,  activation="softmax"),
])
model2.compile(optimizer="adam",
               loss="sparse_categorical_crossentropy",
               metrics=["accuracy"])

# Учим 15 эпох и запоминаем историю обучения
history = model2.fit(X_train, y_train,
                     epochs=15,
                     validation_data=(X_test, y_test),
                     verbose=1)

## Шаг 7. 📈 Ловим переобучение глазами

Построим две линии: точность на train и на test по эпохам.

**Что искать:** сначала линии растут вместе. Потом train продолжает лезть вверх, а test выходит на плато или чуть проседает — **линии расходятся**. Вот это расхождение и есть переобучение: сеть начала «зазубривать» train вместо того, чтобы понимать.

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history.history["accuracy"],     label="train (обучающие)", linewidth=2)
plt.plot(history.history["val_accuracy"], label="test (проверочные)", linewidth=2)
plt.xlabel("Эпоха")
plt.ylabel("Точность")
plt.title("Где train обгоняет test — там начинается переобучение")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# Разрыв между train и test на последней эпохе
gap = history.history["accuracy"][-1] - history.history["val_accuracy"][-1]
print(f"Разрыв train - test на последней эпохе: {gap:.4f}")
print("Чем больше разрыв, тем сильнее сеть 'зазубрила' обучающие данные.")

### 🤔 Твой вывод по эксперименту

Ответь словами:
- На какой примерно эпохе линии начали расходиться?
- Стало ли лучше от второго слоя и 15 эпох, или точность на test почти не изменилась?

In [ ]:
# 📝 Твой вывод по эксперименту:
vyvod_2 = """
...
"""
print(vyvod_2)

---
# 💬 Вопросы для обсуждения

Ответь на них своими словами — можно писать прямо в ячейках или обсудить в классе.

**1. Почему для нейросети картинка — это числа, а не «изображение»?**

**2. Что означает, что softmax даёт вероятности? Чем это полезнее простого ответа?**

**3. Если сеть уверенно ошибается на шуме, можно ли ей полностью доверять в важных задачах (медицина, беспилотники)?**

### 🧪 Проверим вопрос 3 на деле: подаём сети чистый шум

Дадим сети **случайный шум** — не цифру вообще. Что она скажет?

In [ ]:
# Случайный шум 28x28 — никакой цифры здесь нет
noise = np.random.rand(1, 28, 28)

p = model.predict(noise, verbose=0)
answer = p.argmax()
confidence = p.max() * 100

plt.imshow(noise[0], cmap="gray")
plt.title(f"Сеть говорит: это {answer} (уверена на {confidence:.0f}%)")
plt.axis("off")
plt.show()

print("Здесь НЕТ никакой цифры. Но сеть обязана выбрать из 10 и уверенно называет одну.")
print("Вывод: сеть не умеет сказать 'это не цифра'. В медицине и беспилотниках это опасно.")

---
# ℹ️ Проверь себя

Ответь, не подглядывая. Ответы — в скрытой ячейке ниже (разверни, только когда ответил сам).

1. Как нейросеть «видит» картинку?
2. Зачем делить значения пикселей на 255?
3. Что делает слой softmax на выходе?

<details>
<summary>👉 Показать ответы</summary>

1. **Как сеть видит картинку.** Как таблицу чисел 28×28 (яркость каждого пикселя). `Flatten` разворачивает её в ряд из 784 чисел. Никакого «изображения» — только числа.

2. **Зачем делить на 255.** Это нормализация: значения 0–255 сжимаются в 0–1. С маленькими числами в одном масштабе сеть учится быстрее и стабильнее.

3. **Что делает softmax.** Превращает сырые числа последнего слоя в 10 вероятностей (по одной на цифру), которые в сумме дают 100%. Ответ сети — цифра с самой большой вероятностью.

</details>

---
### 🎉 Готово!

Ты обучил нейросеть, разобрал её ошибки, увидел переобучение своими глазами и понял, почему ИИ нельзя доверять слепо.

**Курс Machine Learning для подростков · Урок 14 · MNIST**